# *Лабораторная работа №3*

## Построение бинарного классификатора на основе случайного леса (Теоретическая часть)

Мы продолжаем работу над датасетом с данными о проданных подержанных автомобилях в Германии в 2016 году (см. https://colab.research.google.com/drive/1fl7f7-8WA5GIjZ6qghfiD_2sEaCc1DQH?usp=sharing). В рамках текущей лабораторной необходимо построить модель бинарного классификатора на основе случайного леса, который бы позволил определить была ли машина в ремонте или нет.


Часть из параметров с кратким описанием приведена ниже:

*   n_estimators=100 - Количество агентов (решающих деревьев), из которых состоит лес.
*   criterion='gini' - Критерий разбиения выборки в узловых вершинах дерева.
*   max_depth=None - Максимальная глубина дерева. Максимальное количество условий, которые могут применятся к записи для определения результата.
*   min_samples_split=2 - Минимальное количество примеров, необходимое для осуществления разбиения в узле.
*   min_samples_leaf=1 - Минимальное количество записей в листовой вершине.
*   max_features='sqrt' - Максимальное количество признаков, которое будет передаваться каждому дереву для анализа
*   bootstrap=True - Определяет использование bootstrap-выборок для обучения деревьев. В случае False каждому дереву передается полный датасет.
*   n_jobs=None - Количество потоков, используемое для параллельных вычислений. При значении -1 используется максимальное доступное количество потоков
*   random_state=None - Семя генератора случайных чисел для фиксирования поведения между запусками. При None семя выбирается случайно для каждого запуска.
*   verbose=0 - Определяет уровень детализации отчетов при обучении и прогнозировании.
*   warm_start=False - Обеспечивает возможность дообучать ранее использовавшиеся модели. В случае False каждый раз лес будет обучаться с нуля.
*   class_weight=None - Дает возможность указать соотношение классов в выборке для повышения качества обучения.



Подключим все необходимые для дальнейшей работы библиотеки.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy

from sklearn import ensemble
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

Загрузим базу данных, которая была получена нами в ходе предыдущей работы, т.е. в ней уже проведена определенная работа по очистке данных (подробнее см. лаб1).

In [ ]:
%%capture
!wget -O autos_mod.csv https://www.dropbox.com/scl/fi/t0wr6jge6wwzoff2upd9q/autos_mod_2025.csv?rlkey=sak848n8x96h5gz1383rzck6m&st=klg3z8my&dl=0

Записываем ее в датафрейм и проверим столбцы на наличие пустых значений.

In [ ]:
df = pd.read_csv('autos_mod.csv', encoding='iso-8859-1')
df.isnull().any()

,0
Unnamed: 0,False
price,False
vehicleType,False
yearOfRegistration,False
gearbox,False
powerPS,False
model,False
kilometer,False
fuelType,False
brand,False


Удалим столбец, который возник в процессе переписывания набора данных, так как никакой полезной информации он не несет.

In [ ]:
del df["Unnamed: 0"]

Разобьем весь наш набора данных на тренировочный и тестовый наборы данных.

In [ ]:
values = df['notRepairedDamage']
points = df.drop(['notRepairedDamage'], axis=1)
train_points, test_points, train_values, test_values = train_test_split(points, values, test_size = 0.2, random_state=42)

Создадим достаточно простенькую модель из 10 решающих деревьев и оценим точность ее работы.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=10, random_state=42)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.9020404909931452


С одной стороны, точность работы классификатора, близкая к 90% - хороший показатель. С другой стороны, не будем забывать о том, что распределение классов в нашем датасете неравномерное. Поэтому проведем сравнение с константным классификатором.

In [ ]:
print(accuracy_score(test_values, np.zeros_like(test_values)))

0.8995098039215687


Классификатор, который абсолютно все объекты относит к одному и тому же классу, работает с сопоставимой точностью. В этом и состоит основная сложность работы с несбалансированными наборами данных. Попробуем увеличить количество деревьев в модели.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.9049298581221107


Увеличение количества деревьев положительно повлияло на точность работы классификатора.

Далее следует оценить наличие влияние параметра bootstrap на качество обучения моделей. Исходя из описания процесса формирования bootstrap-выборок, можно сделать вывод, что качество обучения не будет отличаться при изменении количества деревьев в ансамбле при отключении механизма формирования выборок, так как маршрут работы с данными будет идентичным для каждого отдельного дерева.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=10,  random_state=42, bootstrap=False)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.8985931771082417


In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100,  random_state=42, bootstrap=False)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.9004065040650406


Проведенный эксперимент показал ошибочность изначального предположения. Точность работы классификатора растет с увеличением количества деревьев в вансамбле, хоть и абсолютные значения точности заметно снизились.

При отключенном механизме формирования бутстрэп-выборок точность ансамбля из сотни деревьев составляет 90.04%, что меньше, чем точность ансамбля из десяти деревьев с включенным механизмом (90.20%).

Помимо использования механизма формирования бутстрэп-выборок на разнообразие маршрута обучения агентов ансамбля влияет список передаваемых каждому агенту признаков. За данную характеристику отвечает параметр max_features. При передаче ему в качестве значения None, каждому агенту будет передаваться полный набор параметров для обучения, по умолчанию же, используется квадртаный корень из общего числа параметров.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=10,  random_state=42, bootstrap=False, max_features=None)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.8761158935118762


In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100,  random_state=42, bootstrap=False, max_features=None)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.8746413199426112


Падение точности при увеличении количества деревьев свидетельствует о наличии эффекта переобучения. Осталось проверить, как поведет себя модель при наличии механизма бутстрэп-выборок с использованием полного набора признаков.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=10,  random_state=42, bootstrap=True, max_features=None)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.8994898772517137


In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100,  random_state=42, bootstrap=True, max_features=None)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.9036744779212498


На основании проведенных замеров можно сформулировать следующие выводы:

*   Ограничение списка передаваемых признаков и аппарат формирования бутстрэп-выборок - важные механизмы в борьбе с переобучением ансамблевых методов (таких как случайный лес). Их совместное использование предоставляет возможность построения многоагентных ансамблей без риска столкнуться с переобучением.
*   Применение механизма формирования бутстрэп-выборок дает больший вклад в итоговую точность модели, чем использование ограниченного списка признаков в рамках обучения каждого конкретного агента.



Так как датасет изначально является несбалансированным, можно попробовать улучшить работу классификатора, добавив вес каждого классса в выборке. Делается это с помощью параметра class_weight и словаря вида: {метка_класса : доля_объектов_класса_в_выборке}.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100, class_weight = { 0 : 0.1, 1 : 0.9}, random_state=42)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.8998087039693926


В данном случае намеренно была допущена ошибка в виде инвертирования весов классов. При неправильной задаче веса объектов в выборке, точность работы классификатора заметно снижается.

In [ ]:
rf_model = ensemble.RandomForestClassifier(n_estimators=100, class_weight = { 0 : 0.9, 1 : 0.1}, random_state=42)
rf_model.fit(train_points, train_values)
test_predict_rf = rf_model.predict(test_points)
print(accuracy_score(test_values, test_predict_rf))

0.9059461182847123


При установке правильных меток точность выросла относительно невзвешенного случая.

*ПРИМЕЧАНИЕ:* Однако использование взвешивания классов может нести одну неочевидную опасность: Не существует гарантии, что соотношение весов объектов разных классов будет совпадать на обучающей и валидационной выборках.

Увеличение количества деревьев привело к улучшению результата, но от идеала он по-прежнему далек. В качестве альтернативы попробуем использовать аппарат искусственных нейронных сетей.

## Построение бинарного классификатора (Практическая часть)

Вашим заданием в данной лабораторной будет построение бинарного классификатора на основе случайного леса для предсказания возможного ухода клиента из банка и оценка влияния гиперпараметров алгоритма на получаемый результат. Задание основано на следующем датасете https://www.kaggle.com/datasets/willianoliveiragibin/bank-churn-prediction?resource=download.

Ниже приведены инструкции для загрузки тренировочной  и валидационной части датасета. Предварительную обработку данных, в случае необходимости, можно скопировать из выполненной второй работы.

In [ ]:
%%capture
!wget -O bank_train.csv https://www.dropbox.com/scl/fi/rz8xmydm1h3bpgjjm30jd/bank_train.csv?rlkey=rylj5xs1icbfokyabtnqo5gdv&dl=0
!wget -O bank_valid.csv https://www.dropbox.com/scl/fi/05p2v6pw76tjf7c21rcyb/bank_valid.csv?rlkey=ifbgda9mj7ybbgl2txnzjq9g1&dl=0

В качестве отчета к работе необходимо предоставить диаграммы отражающие связь между гиперпараметрами случайного леса (количество деревьев, максимальная глубина каждого дерева и три любых из оставшихся параметров на выбор студента) и получаемой точностью модели.

Для обеспечения стабильности результата между запусками следует использовать random_state = номер_студента_в_группе.